In [ ]:
# Levi Schult 27 July 2026

import enterprise
import enterprise.signals.anis_coefficients as ac
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc
import pickle as pkl
from enterprise_extensions import model_utils, sampler, blocks
from enterprise.signals import gp_signals, parameter, white_signals, utils, gp_priors

## making some useful functions

In [ ]:
def plot_pulsars(dataset, highlight=None, returnphitheta=False):
    '''
    plots pulsar positions from a dataset pkl 
    if you want to save the pulsar positions, set returnphitheta=True
    
    inputs:
    dataset (str): path to dataset
    highlight (list): list of pulsar names as strings

    outputs:
    psrphis (array): phi of pulsars
    psrthetas (array): theta of pulsars

    usage:
    psrphis, psrthetas = pulsarphitheta('location/of/dataset.pkl')
    '''
    # bringing in pulsars 
    try:
        with open(dataset , 'rb') as ff:
            psrs1 = pkl.load(ff)
    except TypeError:
        psrs1 = dataset # LSS if the dataset is already a list of pulsars, just use it directly
    # LSS getting psr locations
    psrphis = np.array([psr.phi for psr in psrs1])
    psrthts = np.array([psr.theta for psr in psrs1])
    # LSS plotting
    hp.visufunc.projscatter(psrthts, psrphis,
                    marker='*',color='white',
                    edgecolors='k',s=100)
    if highlight is not None:
        psrnames = [p.name for p in psrs1] # LSS highlight pulsars based on name
        highlight_idx = []
        for n in highlight:
            highlight_idx.append(psrnames.index(n))
        hp.visufunc.projscatter(np.array(psrthts)[highlight_idx], 
                                np.array(psrphis)[highlight_idx],
                    marker='*',color='yellow',
                    edgecolors='k',s=100)
    if returnphitheta:
        return psrphis, psrthts



def chain_skymap(phi, theta, nside, ct=False):
    '''
    makes an array of pixel values for healpy plotting from sky location posteriors

    inputs:
    phi (array/list): burned chain of phi samples from mcmc analysis
    costheta (array/list): burned chain of costheta samples from mcmc analysis
    nside (int): should be power of 2 - the resolution of output healpy map (12Nside**2)
    ct (bool): default false - if you want to pass in costheta directly, set to True.

    outputs:
    skymap - an array of pixel values to be passed to hp.mollview or other proj.
    '''
    npix = hp.nside2npix(nside)

    # convert to HEALPix indices
    if ct:
        indices = hp.ang2pix(nside, np.arccos(theta), phi)
    else: indices = hp.ang2pix(nside, theta, phi)

    idx, counts = np.unique(indices, return_counts=True)

    # fill the fullsky map
    hpx_map = np.zeros(npix, dtype=float)
    normcts = counts / np.sum(counts) # LSS normalizing
    hpx_map[idx] = normcts

    return hpx_map


## getting some pulsars

In [ ]:
from defiant.extra import mdc1_utils
mdcpsrs, injparams = mdc1_utils.get_MDC1_psrs(use_pickle=True)

mdcpsrs = mdcpsrs

# LSS this is the resolution of our skymaps
nsi=16
npsrs = len(mdcpsrs)

## Constructing ORFs with arbitrary skymaps

Throughout this notebook we will take advantage of a linear algebra approach 
to calculate the sky integral that gives the ORF. This comes from 
[Taylor+2020](http://arxiv.org/abs/2006.04810) and [Pol+22](http://arxiv.org/abs/2206.09936)

To see this briefly, lets remember the integral that gives us HD:

$\Gamma^{HD} = \frac{3}{2} \int_{S^2} \frac{d^2\hat{\Omega}}{4\pi} 
P(\hat{\Omega}) \kappa_{ab}(f, \hat{\Omega})\sum_{A=+,\times}F^A_a(\hat{\Omega})F^A_b(\hat{\Omega})$

where $P(\hat{\Omega}) = 1 \forall \hat{\Omega}$ and $a, b$ range over pulsars

$\kappa_{ab}$ represents the Earth-pulsar term wave interference effects which 
is highly oscillatory across the sky, so we can approximate this as $(1 + \delta_{ab})$

To move into an easier calculation space, we transform our 2 sphere into equal area
pixels, which transforms the above integral to

$\Gamma^{HD} = \frac{3(1+\delta_{ab})}{2 N_{\rm{px}}} \sum_k P_k \left[ F^+_{a, k}F^+_{b,k} + F^{\times}_{a, k} F^{\times}_{b,k} \right]$

The $N_{\rm pix}$ term takes care of the $4 \pi$ sky integral normalization from the original eqn.

This can make the integral quite easy to compute, as we absorb normalization
factors into our response matrix, which has shape Npairs x Npix:

$R = \frac{3(1+\delta_{ab})}{2 N_{\rm{px}}} \left[ F^+_{a, k}F^+_{b,k} + F^{\times}_{a, k} F^{\times}_{b,k} \right]$

$\Gamma = R P$


### Lets first use this to get HD

In [ ]:
# LSS make an isotropic sky:
isomap = np.ones(hp.nside2npix(nsi))
isomap = (isomap/np.sum(isomap)) # LSS normalize to 1
hp.mollview(isomap, rot=180)

In [ ]:
# LSS construct the response matrix for the pulsars:
# NOTE: conventions are important! in the signalresponse functions in enterprise,
# the gwtheta and phi are interpreted as the propagation direction. 
# However, when we make skymaps we are usually making the opposite (source direction)

# LSS get pixel vectors
npixels = hp.nside2npix(nsi)
pixels = hp.pix2ang(nsi, np.arange(npixels), nest=False)
pixvec = hp.ang2vec(pixels[0], pixels[1])
# LSS invert the pixel vectors to get the source direction.
invvec = -pixvec
gwtheta, gwphi = hp.vec2ang(invvec)

# LSS get the pulsar sky locations
psrtht = np.array([p.theta for p in mdcpsrs])
psrphi = np.array([p.phi for p in mdcpsrs])

# LSS calculate signal response
# LSS note: this includes the 3/2 and 1/npixels normalization factors.
F_e = ac.signalResponse_fast(psrtht, psrphi, gwtheta, gwphi)
# LSS this response matrix has plus and cross interleaved
Fp = F_e[:, 0::2]
Fc = F_e[:, 1::2]

a, b = np.triu_indices(len(mdcpsrs))
R = np.zeros((npsrs, npsrs, npixels))
R[a, b, :] = Fp[a,:]*Fp[b,:] + Fc[a,:]*Fc[b,:]
R[b, a, :] = R[a, b, :] # LSS filling bottom triangle of matrix
idx = np.arange(npsrs)
# LSS doubling diagonal to take care of pulsar term
R[idx, idx, :] = R[idx, idx, :]*2
# LSS R is now npsr, npsr, npix shape

In [ ]:
# LSS make HD from this response matrix

### FILL IN HERE! ####
gam_HD = R @ isomap

print(npsrs)
print(gam_HD.shape)

# LSS get ang sep:
psr_pos = np.array([p.pos for p in mdcpsrs])
xi = np.arccos(np.sum(psr_pos[a]*psr_pos[b], axis=1))

# LSS plot the ORF
# LSS mult by npixels to get the normalization we are used to for HD
plt.plot(xi, gam_HD[a, b]*npixels, '.')
plt.axhline(0, color='k', ls='--')

##### make map of array response matrix

In [ ]:
Rmap = np.sum(np.abs(R), axis=(0, 1))

In [ ]:
hp.mollview(Rmap, rot=180)
plot_pulsars(mdcpsrs)

In [ ]:
# LSS lets make a function of the above for quick plotting 
# and comparison

def plotHD():
    # LSS plot the ORF
    plt.plot(xi, gam_HD[a, b]*npixels, '.', label='HD')
    plt.axhline(0, color='k', ls='--')
    plt.legend()

def plot_orf(sky_map, label=None):
    gam_cosvar = R @ sky_map
    plt.plot(xi, gam_cosvar[a, b]*npixels, 'o', label=label)

In [ ]:
# LSS lets get our statistically isotropic skymap:
p_cosvar = np.square(np.random.rayleigh(np.sqrt(1/2), size=npixels))
p_cosvar = p_cosvar/np.sum(p_cosvar) # LSS normalize to 1
hp.mollview(p_cosvar, title='Isotropic Skymap', rot=180)
plot_pulsars(mdcpsrs)
plt.show()

In [ ]:
for i in range(100):
    p_cosvar = np.square(np.random.rayleigh(np.sqrt(1/2), size=npixels))
    p_cosvar = p_cosvar/np.sum(p_cosvar) # LSS normalize to 1
    plot_orf(p_cosvar)

plotHD()
plt.show()

## making some anisotropic bases

### linear spherical harmonic

Here we decompose the power on the sky $P(\hat{\Omega})$ as a sum of spherical harmonics.


$P(\hat{\Omega}) = \sum_l \sum_{m=-l}^{+l} c_{lm} Y_{lm}$

In [ ]:
# Let's build our anisotropic basis functions for the ORF. 
# for this we need the pulsar locations in phi and theta.
lmx = 2
nclms = (lmx+1)**2

psrlocs = np.array([psrphi, psrtht]).T

linspharm_basis = ac.anis_basis(psrlocs, lmax=lmx, nside=16)

##### What did this do?


this gives us linear spherical harmonic basis functions for the ORF
shaped as nclms, npsr, npsr.

nclms is determined by our lmax. We always set c00=sqrt(4pi) for normalization
purposes since it controls the isotropic component (this is what we are modeling with A/gamma or free spectral parameters)

so, $N_{c_{lm}} = (l_{max}+1)^2$

Now, if we have a vector of clm values, we can multiply them by these basis functions to get the corresponding ORF

In [ ]:
print((lmx+1)**2, npsrs, npsrs)
linspharm_basis.shape

##### lets see some of these basis functions

clms are ordered like this:

$c_{lm} = c_{0,0}, c_{1, -1}, c_{1, 0}, c_{1, 1}, c_{2, -2}, c_{2, -1} ....$

Here is where things get a little confusing with repeated terminology.

In spherical harmonics, the modes are often referred to by their polar signature e.g.

$l=0$ is the monopolar component
- python index: 0 - always set to $\sqrt{4\pi}$

$l=1$ is the dipolar component
- python indices: 1, 2, 3

$l=2$ is the quadrupolar component
- python indices: 4, 5, 6, 7, 8

$l=3$ is the octopolar component


In [ ]:
# LSS lets look at the dipolar component
clms = np.zeros(nclms)
clms[0] = np.sqrt(4*np.pi)

### FILL IN HERE! ####
# clms[] = 1.0

smap = ac.mapFromClm_fast(clms, nside=nsi)
smap = smap / np.sum(smap)
hp.mollview(smap, title=r'$c_{1, -1}=1$', rot=180)
plot_pulsars(mdcpsrs)
plt.show()

# LSS here is what the ORF looks like for this skymap:
plot_orf(smap, label='Dipole')
plotHD()

Why is there spread at each angular separation? This is because of the anisotropy!

Some pulsar pairs are located on parts of sky with more GW power or less, so their correlations will be slightly different!

In [ ]:
# LSS lets look at the quadrupolar component
clms = np.zeros(nclms)
clms[0] = np.sqrt(4*np.pi)

### FILL IN HERE! ####
#clms[4] = 0

smap = ac.mapFromClm_fast(clms, nside=nsi)
smap = smap / np.sum(smap)
hp.mollview(smap, title=r'$c_{2, -2}=1$', rot=180)
plot_pulsars(mdcpsrs)
plt.show()

# LSS here is what the ORF looks like for this skymap:
plot_orf(smap, label='Quadrupole')
plotHD()

- remember: we are modeling the sky via a combination of these basis functions, hopefully enabling us to capture GW hotspots from individual SMBHBs

In [ ]:
# Here is a space where you can put in whatever coefficients you want!
# what does the sky look like? What does the ORF look like?
# I would recommend staying within -5, 5 for the clms 
# you could use np.random.uniform to generate these values.

clms = np.zeros(nclms)
clms[0] = np.sqrt(4*np.pi)
### FILL IN HERE! ####
#clms[1:] = 0

smap = ac.mapFromClm_fast(clms, nside=nsi)
smap = smap / np.sum(smap)
hp.mollview(smap, title='random clms', rot=180)
plot_pulsars(mdcpsrs)
plt.show()

# LSS here is what the ORF looks like for this skymap:
plot_orf(smap, label='rand clms')
plotHD()


The resolution of our GW hotspots is dependent on our chosen $l_{max}$. Let's 
increase it to see our map get finer details


In [ ]:
# LSS try changing the lmax to see how detailed the sky becomes!

### FILL IN HERE! ####
#lmx = 
nclms = (lmx+1)**2

psrlocs = np.array([psrphi, psrtht]).T

linspharm_basis = ac.anis_basis(psrlocs, lmax=lmx, nside=16)

clms = np.zeros(nclms)
clms[0] = np.sqrt(4*np.pi)
clms[1:] = np.random.uniform(-5, 5, size=nclms-1)

smap = ac.mapFromClm_fast(clms, nside=nsi)
smap = smap / np.sum(smap)
hp.mollview(smap, title=r'random clms, $l_{max} = $' + f'{lmx}', rot=180)
plot_pulsars(mdcpsrs)
plt.show()

# LSS here is what the ORF looks like for this skymap:
plot_orf(smap, label='rand clms')
plotHD()

In [ ]:
print(f'Number of parameters for lmax={lmx} : {nclms-1}')

Neato! There are problems with the linear spherical harmonic basis though. You might 
have noticed that the skymap has negative power on some parts of the sky. This is unphysical.
There should be positive GW power everywhere on the sky. Analyses in the past 
have handled this by enforcing a physical prior during an MCMC. This means that 
any proposed set of $c_{lm}$s was checked to produce a positive sky. If they 
didn't then the clms were rejected.

A solution to this is modeling the sqrt of the sky's power instead. This way it
will always be positive. This is the sqrt spherical harmonic model:

$\left[ P(\hat{\Omega}) \right]^{1/2} = b_{lm} Y_{lm}$

we use $b_{lm}$ to differentiate between linear and sqrt spharms.


Additionally, the number of parameters explodes as we try to probe finer 
angular scales, which is a problem since we expect an individual SMBHB to make 
very small scale anisotropies. There are some complicated PTA angular resolution
aspects that make the binary's induced anisotropy slightly different than a 
point sourcebut the argument remains.



## Radiometer Basis

Here we assert that all GW power on the sky is coming from a specific point. This
gives us a specific ORF for that sky position. 

We can use this for anisotropy searches in 2 ways:

1. **Radiometer Search**: In this analysis, we use the data to constrain how much 
power is coming from a single pixel and then repeat this for every pixel on the
sky. This creates a skymap of GW power based on what the data supports for the
amount of power in each pixel, but it requires a LOT (N_pixels) of MCMC runs. 

2. **Pixel Search**: In this approach, we give our model theta and phi (sky position)
parameters. When points are proposed, it pulls the ORF corresponding to that sky
position. This tells us where the data most supports excess GW power on the sky,
be it a hotspot or very unconstrained regions over the sky. This enables us to 
have decent angular resolution with few parameters.

A downside of the radiometer basis is that the pixels are not correlated.

In [ ]:
# to see this radiometer ORF, it's actually within the response function we made
# before:

### FILL IN HERE! ####
randompixel = # Choose a random pixel! could use np.random.randint 

smap = np.zeros(npixels)
smap[randompixel] = npixels
smap = smap / np.sum(smap)
hp.mollview(smap, title='Loud Pixel', rot=180)
plot_pulsars(mdcpsrs)
plt.show()

plot_orf(smap, label=f'Radiometer ORF for pixel:{randompixel}')
plotHD()

In [ ]:
# sure enough, lets plot the orf matrix that corresponds to this pixel:
### FILL IN HERE! ####
plt.plot(xi, R[a, b, ___]*npixels, 'o', label=f'R for pixel {randompixel}')
plotHD()

## Let's do a very basic anisotropy analysis via the pixel search

In [ ]:
# defining in enterprise format

from enterprise.signals import signal_base

@signal_base.function
def radiometer_orf(pos1, pos2, psr_pos_map_dict, radiometer_corr, costheta, phi, nside):
    theta = np.arccos(costheta) # correcting for sampling in costheta
    phi = phi
    psr1_index = psr_pos_map_dict[pos1[0]]
    psr2_index = psr_pos_map_dict[pos2[0]]
    sourcedir = hp.ang2vec(theta, phi) # convert to vector
    pidx = hp.vec2pix(nside=nside, x=sourcedir[0], y=sourcedir[1], z=sourcedir[2]) # convert to pixel again
    return radiometer_corr[psr1_index, psr2_index, pidx]


In [ ]:
# LSS lets import a dataset I injected with a LOUD CW
psrs = pkl.load(open('./levi_mdc_style_cw.pkl', 'rb'))



# LSS get pixel vectors
npixels = hp.nside2npix(nsi)
pixels = hp.pix2ang(nsi, np.arange(npixels), nest=False)
pixvec = hp.ang2vec(pixels[0], pixels[1])
# LSS invert the pixel vectors to get the source direction.
invvec = -pixvec
gwtheta, gwphi = hp.vec2ang(invvec)

# LSS get the pulsar sky locations
psrtht = np.array([p.theta for p in psrs])
psrphi = np.array([p.phi for p in psrs])

# LSS calculate signal response
# LSS note: this includes the 3/2 and 1/npixels normalization factors.
F_e = ac.signalResponse_fast(psrtht, psrphi, gwtheta, gwphi)
# LSS this response matrix has plus and cross interleaved
Fp = F_e[:, 0::2]
Fc = F_e[:, 1::2]

a, b = np.triu_indices(len(mdcpsrs))
R = np.zeros((npsrs, npsrs, npixels))
R[a, b, :] = Fp[a,:]*Fp[b,:] + Fc[a,:]*Fc[b,:]
R[b, a, :] = R[a, b, :] # LSS filling bottom triangle of matrix
idx = np.arange(npsrs)
# LSS doubling diagonal to take care of pulsar term
R[idx, idx, :] = R[idx, idx, :]*2
# LSS R is now npsr, npsr, npix shape


psrs_pos = np.array([p.pos for p in psrs])

psr_pos_map_dict = {}

for pos1 in psrs_pos:
    for ii in range(len(psrs_pos)):

        if np.all(psrs_pos[ii] == pos1):
            psr_pos_map_dict.update({pos1[0]: ii})


In [ ]:
# LSS set up our model
comps = 1

Tspan = model_utils.get_tspan(psrs)

# timing model
tm = gp_signals.MarginalizingTimingModel(use_svd=True)

efac = parameter.Constant(1.0)
wn = white_signals.MeasurementNoise(efac=efac)

# LSS building an anisotropic free spec model.
# pixorf = radiometer_orf(psr_pos_map_dict=psr_pos_map_dict, nside=nsi, radiometer_corr = R, costheta = parameter.Uniform(-1,1.)('costheta'), phi = parameter.Uniform(0, 2*np.pi)('phi'))
# pixaniso = blocks.common_red_noise_block(psd='spectrum', Tspan=Tspan, orf=radiometer_orf(psr_pos_map_dict=psr_pos_map_dict, nside=nsi, radiometer_corr = R, costheta = parameter.Uniform(-1,1.)('costheta'), phi = parameter.Uniform(0, 2*np.pi)('phi')),
#                                           components=comps, name='pixelsearch')


rho_gw = parameter.Uniform(-9, -4, size=1)('rho_gw')

fs = gp_priors.free_spectrum(log10_rho=rho_gw)

pixaniso = gp_signals.FourierBasisCommonGP(fs, orf = radiometer_orf(psr_pos_map_dict=psr_pos_map_dict, nside=nsi, radiometer_corr = R, costheta = parameter.Uniform(-1,1.)('costheta'), phi = parameter.Uniform(0, 2*np.pi)('phi')), components=comps, Tspan=Tspan, name='pixelsearch')


In [ ]:
# LSS assemble the model:

### FILL IN HERE! ####
#s = ___

models = [s(p) for p in psrs]

pta = signal_base.PTA(models)

In [ ]:
pta.params

In [ ]:
cov = np.eye(len(pta.params))*0.01

mcmc = ptmcmc(ndim=len(pta.params), cov=cov, logl=pta.get_lnlikelihood, 
              logp=pta.get_lnprior, outDir='./pixelsearch_demo/', resume=False)

d0 = parameter.sample(pta.params)

x1 = np.hstack([d0[par.name] for par in pta.params])

In [ ]:
d0

In [ ]:
x1

In [ ]:
mcmc.sample(x1, int(1e4), SCAMweight=25, AMweight=25, DEweight=50)

In [ ]:
chn = np.loadtxt('./pixelsearch_demo/chain_1.txt')

In [ ]:
# Let's look at the trace plot - how fuzzy does it look?
# You will probably need to trim off the burn in to see this.
# a reasonable burn in is 25% of chain length
plt.plot(chn[:, -4])

In [ ]:
### FILL IN HERE! ####
# take out burn in before giving the chain to the function.

# smap = chain_skymap(theta=chn[:, 0], phi=chn[300:, 1], nside=nsi, ct=True)

In [ ]:
# CW params
gwtheta = np.pi/2.
gwphi = np.pi

hp.mollview(smap, title='Pixel Search Skymap', rot=180)
plot_pulsars(psrs)
hp.projscatter(gwtheta, gwphi, marker='x', color='red', s=100)

### Cosmic Variance

We also know the sky won't look completely isotropic though. The HD is an average over many universes.

Just as an illustrative example, we have made a statistically isotropic sky below.
A full simulation of cosmic variance is quite complicated and beyond this tutorial.

In [ ]:
# LSS lets get our statistically isotropic skymap:
p_cosvar = np.square(np.random.rayleigh(np.sqrt(1/2), size=npixels))
p_cosvar = p_cosvar/np.sum(p_cosvar) # LSS normalize to 1
hp.mollview(p_cosvar, title='Isotropic Skymap', rot=180)
plot_pulsars(mdcpsrs)
plt.show()

In [ ]:
for i in range(100):
    p_cosvar = np.square(np.random.rayleigh(np.sqrt(1/2), size=npixels))
    p_cosvar = p_cosvar/np.sum(p_cosvar) # LSS normalize to 1
    plot_orf(p_cosvar)

plotHD()
plt.show()